In [6]:
import pandas as pd
import ir_datasets
from src.data import DATA_DIR_INTERIM
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels

from src.data import load_qrels_from_path
from src.utils import format_score

from topic_gen import logger
logger.setLevel("DEBUG")

In [7]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_INTERIM / "qrels-dl21-reference"
predictions, names, metadata = load_qrels_from_path(BASE_DIR)

# binarize qrels
predictions = [binarize_qrels(qrels) for qrels in predictions]

In [8]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "msmarco-document-v2/trec-dl-2021/judged").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 8/3992 qrels in references but not in predictions.


[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 11/3989 qrels in references but not in predictions.


In [10]:
df = pd.DataFrame(res)
df["score"] = df.apply(format_score, axis=1)
df = df.pivot(index="name", columns="measure", values="score").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")

df[["date", "model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,date,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,2025-11-26_16:48:16,qwen3-30B-no-think,-DNA-zero-shot-dl,0.20 ± 0.03,0.22 ± 0.01,0.83 ± 0.03
1,2025-11-26_17:03:56,qwen3-30B-no-think,-DNA-zero-shot-dl,0.23 ± 0.02,0.21 ± 0.01,0.84 ± 0.02
